# Agent Controller (Phase 5: MQTT Subscriber + Publisher)

This notebook subscribes to sheep/wolf/grass/tick streams and publishes per-tick control commands.

Phase 5 scope:
- Subscribe: `.../sim/tick`, `.../sim/sheep/state`, `.../sim/wolf/state`, `.../sim/grass/state`
- Publish: `.../sim/controller/commands`

In [1]:
# Cell purpose: import dependencies and connect to MQTT using project config.
from datetime import datetime, timezone
import json
import time

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="agent-controller-phase5")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

publisher = MqttPublisher(connector)
print("STARTUP_OK: agent_controller connected and ready")

Loaded config. Primary MQTT profile: 127.0.0.1:1883
MQTT connected: True
STARTUP_OK: agent_controller connected and ready


In [2]:
# Cell purpose: define topics, thresholds, and controller state cache.
base_topic = config.mqtt.base_topic.rstrip("/")
tick_topic = f"{base_topic}/sim/tick"
sheep_state_topic = f"{base_topic}/sim/sheep/state"
wolf_state_topic = f"{base_topic}/sim/wolf/state"
grass_state_topic = f"{base_topic}/sim/grass/state"
controller_commands_topic = f"{base_topic}/sim/controller/commands"

simulation_cfg = config.simulation

min_sheep_threshold = simulation_cfg.min_sheep_threshold if simulation_cfg is not None else 12
max_wolf_threshold = simulation_cfg.max_wolf_threshold if simulation_cfg is not None else 25
low_grass_threshold_pct = simulation_cfg.low_grass_threshold_pct if simulation_cfg is not None else 25
max_ticks = simulation_cfg.max_ticks if simulation_cfg is not None else 1000
extinction_grace_ticks = simulation_cfg.extinction_grace_ticks if simulation_cfg is not None else 20

default_sheep_repro_prob = (
    simulation_cfg.sheep_reproduction_probability if simulation_cfg is not None else 0.08
)
default_wolf_repro_prob = (
    simulation_cfg.wolf_reproduction_probability if simulation_cfg is not None else 0.04
)
default_sheep_move_cost = simulation_cfg.sheep_move_cost if simulation_cfg is not None else 1
default_wolf_move_cost = simulation_cfg.wolf_move_cost if simulation_cfg is not None else 1

latest_sheep_count = simulation_cfg.initial_sheep if simulation_cfg is not None else 30
latest_wolf_count = simulation_cfg.initial_wolves if simulation_cfg is not None else 8
latest_grass_pct = simulation_cfg.initial_grass_coverage_pct if simulation_cfg is not None else 70
latest_tick = 0
extinction_streak = 0
stop_published = False

print("Subscribe topics:")
print(f"- {tick_topic}")
print(f"- {sheep_state_topic}")
print(f"- {wolf_state_topic}")
print(f"- {grass_state_topic}")
print("Publish topics:")
print(f"- {controller_commands_topic}")

Subscribe topics:
- simulated-city/sim/tick
- simulated-city/sim/sheep/state
- simulated-city/sim/wolf/state
- simulated-city/sim/grass/state
Publish topics:
- simulated-city/sim/controller/commands


In [3]:
# Cell purpose: consume state updates and publish control commands each tick.
def on_message(client, userdata, msg):
    global latest_sheep_count, latest_wolf_count, latest_grass_pct
    global latest_tick, extinction_streak, stop_published

    try:
        payload = json.loads(msg.payload.decode("utf-8"))
    except Exception:
        print(f"Skipping non-JSON payload on topic {msg.topic}")
        return

    if msg.topic == sheep_state_topic:
        latest_sheep_count = int(payload.get("sheep_count", latest_sheep_count))
        return

    if msg.topic == wolf_state_topic:
        latest_wolf_count = int(payload.get("wolf_count", latest_wolf_count))
        return

    if msg.topic == grass_state_topic:
        latest_grass_pct = float(payload.get("grass_coverage_pct", latest_grass_pct))
        return

    if msg.topic == tick_topic:
        tick = int(payload.get("tick", latest_tick + 1))
        latest_tick = tick

        if latest_sheep_count == 0 or latest_wolf_count == 0:
            extinction_streak += 1
        else:
            extinction_streak = 0

        sheep_repro_prob = default_sheep_repro_prob
        wolf_repro_prob = default_wolf_repro_prob
        sheep_move_cost = default_sheep_move_cost
        wolf_move_cost = default_wolf_move_cost
        reasoning = []

        if latest_sheep_count < min_sheep_threshold and latest_wolf_count > max_wolf_threshold:
            wolf_repro_prob = max(0.0, round(default_wolf_repro_prob * 0.5, 4))
            reasoning.append("sheep_low_wolf_high")

        if latest_grass_pct < low_grass_threshold_pct:
            sheep_repro_prob = max(0.0, round(default_sheep_repro_prob * 0.5, 4))
            sheep_move_cost = default_sheep_move_cost + 1
            reasoning.append("grass_low")

        stop_requested = tick >= max_ticks or extinction_streak >= extinction_grace_ticks
        if stop_requested:
            reasoning.append("stop_condition")

        command_payload = {
            "tick": tick,
            "apply_next_tick": tick + 1,
            "sheep_reproduction_probability": sheep_repro_prob,
            "wolf_reproduction_probability": wolf_repro_prob,
            "sheep_move_cost": sheep_move_cost,
            "wolf_move_cost": wolf_move_cost,
            "stop_requested": stop_requested,
            "reason": reasoning if reasoning else ["baseline"],
            "snapshot": {
                "sheep_count": latest_sheep_count,
                "wolf_count": latest_wolf_count,
                "grass_coverage_pct": latest_grass_pct,
                "extinction_streak": extinction_streak,
            },
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        publisher.publish_json(controller_commands_topic, json.dumps(command_payload), qos=0)

        if stop_requested and not stop_published:
            stop_published = True
            print(
                f"tick={tick} command=STOP sheep={latest_sheep_count} wolves={latest_wolf_count} "
                f"grass={latest_grass_pct}% streak={extinction_streak} | published"
            )
        else:
            print(
                f"tick={tick} command=UPDATE sheep_prob={sheep_repro_prob} "
                f"wolf_prob={wolf_repro_prob} sheep_cost={sheep_move_cost} "
                f"wolf_cost={wolf_move_cost} reason={command_payload['reason']} | published"
            )

connector.client.on_message = on_message
connector.client.subscribe(tick_topic, qos=0)
connector.client.subscribe(sheep_state_topic, qos=0)
connector.client.subscribe(wolf_state_topic, qos=0)
connector.client.subscribe(grass_state_topic, qos=0)
print("Subscriptions active.")

Subscriptions active.


In [ ]:
# Cell purpose: keep this notebook alive to process incoming MQTT messages.
print("Controller agent loop running. Interrupt the cell to stop.")
try:
    while True:
        time.sleep(0.2)
except KeyboardInterrupt:
    print("Stopping controller agent loop.")
finally:
    connector.disconnect()
    print("MQTT disconnected.")

Controller agent loop running. Interrupt the cell to stop.


tick=47 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=48 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=49 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=50 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=51 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=52 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=53 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=54 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
tick=55 command=UPDATE sheep_prob=0.08 wolf_prob=0.04 sheep_cost=1 wolf_cost=1 reason=['baseline'] | published
t